Vanilla RAG Framework for course notes.
author: Pearl Park

In [1]:
from dotenv import load_dotenv

load_dotenv()  


True

In [15]:
# import chroma for making embeddings:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = PyPDFLoader("data/cloud_computing.pdf")
pages = loader.load()
print(pages[0].metadata)

{'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'CPSC 436C', 'source': 'data/cloud_computing.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}


In [16]:
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap= 100,
) 

chunks = pdf_splitter.split_documents(pages)
print(len(chunks))
print(chunks[10].page_content[:200])
print(chunks[10].metadata)


124
Tasks
 
change
 
pattern:
 
 
●
 
Training
 
model
 
on
 
all
 
rows
 
means
 
nothing
 
is
 
skipped;
 
whole
 
file
 
has
 
to
 
be
 
read
 
 
●
 
Model
 
choice
 
random
 
forest
 
(independent
 
t
{'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'CPSC 436C', 'source': 'data/cloud_computing.pdf', 'total_pages': 27, 'page': 2, 'page_label': '3'}


In [17]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_vertexai import VertexAIEmbeddings, ChatVertexAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os

PROJECT = os.environ["GCP_PROJECT"]

# embeddings + vector store 
embeddings = VertexAIEmbeddings(model_name="text-embedding-005", project=PROJECT)
vector_store = Chroma(collection_name="lectures", embedding_function=embeddings,
                      persist_directory="vectorstore")

BATCH_SIZE=20
for i in range(0, len(chunks), BATCH_SIZE):
    vector_store.add_documents(chunks[i:i+BATCH_SIZE])


# retriever + LLM
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
llm = ChatVertexAI(model_name="gemini-2.5-flash", project=PROJECT, location="us-central1")

# format docs with citations
def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata['source'].split('/')[-1]}, page {d.metadata['page'] + 1}]\n{d.page_content}"
        for d in docs
    )

/var/folders/kx/g15cvyfn56z4p94t1fz7sjpc0000gn/T/ipykernel_7644/3853657737.py:14: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(model_name="text-embedding-005", project=PROJECT)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/var/folders/kx/g15cvyfn56z4p94t1fz7sjpc0000gn/T/ipykernel_7644/3853657737.py:25: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model_name="gemini-2.5-flash", project=PROJECT, location="us-central1")


Learning Spark calls transformations that need rows from other partitions "wide transformations" ([l6.pdf, page 11]). For these transformations, Spark moves data first, which is referred to as "Exchange" ([l6.pdf, page 11]).

Examples of wide transformations include:
*   Totals per key (groupBy, GROUP BY) ([l6.pdf, page 11])
*   Join of two large tables ([l6.pdf, page 11])
*   Global sort (orderBy) ([l6.pdf, page 11])
*   Value per key written back on every row (window) ([l6.pdf, page 11])
I don't know.


In [ ]:
prompt = PromptTemplate.from_template("""Use only the context to answer the question.
Cite the file and page for each point. If the answer isn't in the context, say you don't know.

context: {context}

question: {query}

answer: """)

# create the rag_chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is a wide transformation in Spark?"))
print(rag_chain.invoke("What is the purpose of life?"))   # should say it doesn't know